# Friedman Statistical Analysis — CNN Attention Mechanisms

This notebook reproduces the formal statistical comparison among the five evaluated configurations:

- Baseline CNN
- CNN + CBAM
- CNN + CoordAttention
- CNN + Triplet Attention
- CNN + ECA-Net

Each configuration was evaluated in five training runs. Runs are matched by run/seed index across configurations (R1 with R1, ..., R5 with R5), so the statistical comparison is treated as a repeated-measures design.

The primary test is the **non-parametric Friedman test**, applied independently to Accuracy, malignant-class Precision, malignant-class Recall, malignant-class F1-score, and ROC-AUC.

- Null hypothesis: the rank distributions are equal across the five configurations.
- Significance level: $\alpha=0.05$.
- Kendall's $W$ is reported as an effect-size measure.
- Pairwise Wilcoxon signed-rank tests with Holm correction are only performed if the corresponding omnibus Friedman test is significant.

The notebook is intentionally **standalone**: it contains the per-run metrics used in the manuscript, so it does not depend on executing the 25 training notebooks first.

## Data provenance

The values below correspond to the five reported runs for each model configuration. Accuracy and malignant-class Precision/Recall/F1 are stored at the numerical precision used for the statistical analysis. ROC-AUC values correspond to the values reported for the five runs in the manuscript.

For repository organization, the original training notebooks can remain as the source experiments, while this notebook serves as the reproducible statistical-analysis notebook.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

pd.set_option("display.precision", 6)

In [2]:
data = {'Baseline CNN': {'Accuracy': [0.7397260274, 0.7123287671, 0.8082191781, 0.7534246575, 0.7808219178], 'Precision': [0.6060606061, 0.5862068966, 0.7142857143, 0.6666666667, 0.7777777778], 'Recall': [0.7692307692, 0.6538461538, 0.7692307692, 0.6153846154, 0.5384615385], 'F1-score': [0.6779661017, 0.6181818182, 0.7407407407, 0.64, 0.6363636364], 'ROC-AUC': [0.85, 0.773, 0.851, 0.792, 0.843]}, 'CBAM': {'Accuracy': [0.7534246575, 0.698630137, 0.7534246575, 0.7260273973, 0.7123287671], 'Precision': [0.7, 0.5909090909, 0.6176470588, 0.625, 0.5714285714], 'Recall': [0.5384615385, 0.5, 0.8076923077, 0.5769230769, 0.7692307692], 'F1-score': [0.6086956522, 0.5416666667, 0.7, 0.6, 0.6557377049], 'ROC-AUC': [0.822, 0.73, 0.856, 0.799, 0.776]}, 'CoordAttention': {'Accuracy': [0.6849315068, 0.6712328767, 0.7260273973, 0.7945205479, 0.8493150685], 'Precision': [0.5555555556, 0.5357142857, 0.6071428571, 0.72, 0.8571428571], 'Recall': [0.5769230769, 0.5769230769, 0.6538461538, 0.6923076923, 0.6923076923], 'F1-score': [0.5660377358, 0.5555555556, 0.6296296296, 0.7058823529, 0.7659574468], 'ROC-AUC': [0.691, 0.773, 0.774, 0.817, 0.856]}, 'Triplet Attention': {'Accuracy': [0.8082191781, 0.7671232877, 0.7123287671, 0.7397260274, 0.7945205479], 'Precision': [0.75, 0.7368421053, 0.5806451613, 0.7058823529, 0.72], 'Recall': [0.6923076923, 0.5384615385, 0.6923076923, 0.4615384615, 0.6923076923], 'F1-score': [0.72, 0.6222222222, 0.6315789474, 0.5581395349, 0.7058823529], 'ROC-AUC': [0.864, 0.831, 0.786, 0.783, 0.804]}, 'ECA-Net': {'Accuracy': [0.8082191781, 0.7808219178, 0.7808219178, 0.7945205479, 0.7808219178], 'Precision': [0.8, 0.6923076923, 0.6785714286, 0.7619047619, 0.6470588235], 'Recall': [0.6153846154, 0.6923076923, 0.7307692308, 0.6153846154, 0.8461538462], 'F1-score': [0.6956521739, 0.6923076923, 0.7037037037, 0.6808510638, 0.7333333333], 'ROC-AUC': [0.839, 0.805, 0.828, 0.844, 0.847]}}

models = list(data.keys())
metrics = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]

rows = []
for model in models:
    for run in range(5):
        row = {"Model": model, "Run": run + 1}
        for metric in metrics:
            row[metric] = data[model][metric][run]
        rows.append(row)

df = pd.DataFrame(rows)
df

,Model,Run,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Baseline CNN,1,0.739726,0.606061,0.769231,0.677966,0.850
1,Baseline CNN,2,0.712329,0.586207,0.653846,0.618182,0.773
2,Baseline CNN,3,0.808219,0.714286,0.769231,0.740741,0.851
3,Baseline CNN,4,0.753425,0.666667,0.615385,0.640000,0.792
4,Baseline CNN,5,0.780822,0.777778,0.538462,0.636364,0.843
5,CBAM,1,0.753425,0.700000,0.538462,0.608696,0.822
6,CBAM,2,0.698630,0.590909,0.500000,0.541667,0.730
7,CBAM,3,0.753425,0.617647,0.807692,0.700000,0.856
8,CBAM,4,0.726027,0.625000,0.576923,0.600000,0.799
9,CBAM,5,0.712329,0.571429,0.769231,0.655738,0.776


In [3]:
# Sanity checks: five models, five matched runs per model, no missing values.
assert len(models) == 5
assert set(df.groupby("Model")["Run"].nunique()) == {5}
assert not df[metrics].isna().any().any()

print("Data checks passed.")
print(f"Models: {len(models)}")
print(f"Runs per model: {df.groupby('Model')['Run'].nunique().unique()[0]}")

Data checks passed.
Models: 5
Runs per model: 5


In [4]:
# Descriptive mean ± sample standard deviation (SD) across the five runs.
summary_rows = []

for model in models:
    sub = df[df["Model"] == model].sort_values("Run")
    row = {"Model": model}
    for metric in metrics:
        row[f"{metric} mean"] = sub[metric].mean()
        row[f"{metric} SD"] = sub[metric].std(ddof=1)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary

,Model,Accuracy mean,Accuracy SD,Precision mean,Precision SD,Recall mean,Recall SD,F1-score mean,F1-score SD,ROC-AUC mean,ROC-AUC SD
0,Baseline CNN,0.758904,0.037012,0.670200,0.078578,0.669231,0.100295,0.662650,0.048770,0.8218,0.036629
1,CBAM,0.728767,0.024505,0.620997,0.049069,0.638462,0.140266,0.621220,0.059862,0.7966,0.047537
2,CoordAttention,0.745205,0.075405,0.655111,0.133693,0.638462,0.058330,0.644613,0.090565,0.7822,0.061496
3,Triplet Attention,0.764384,0.039227,0.698674,0.068060,0.615385,0.108786,0.647565,0.066224,0.8136,0.034034
4,ECA-Net,0.789041,0.012252,0.715969,0.063005,0.700000,0.095768,0.701170,0.019766,0.8326,0.017038


In [5]:
# Friedman omnibus tests.
n_runs = 5
k_models = 5
friedman_rows = []

for metric in metrics:
    samples = [
        df[df["Model"] == model].sort_values("Run")[metric].to_numpy()
        for model in models
    ]

    chi2, p = friedmanchisquare(*samples)
    kendall_w = chi2 / (n_runs * (k_models - 1))

    friedman_rows.append({
        "Metric": metric,
        "Friedman chi-square": chi2,
        "df": k_models - 1,
        "p-value": p,
        "Kendall W": kendall_w,
        "Significant at 0.05": p < 0.05,
    })

friedman_results = pd.DataFrame(friedman_rows)
friedman_results

,Metric,Friedman chi-square,df,p-value,Kendall W,Significant at 0.05
0,Accuracy,4.783505,4,0.310241,0.239175,False
1,Precision,3.680000,4,0.451041,0.184000,False
2,Recall,3.673469,4,0.451996,0.183673,False
3,F1-score,4.960000,4,0.291426,0.248000,False
4,ROC-AUC,2.222222,4,0.694963,0.111111,False


In [6]:
# Compact table matching the manuscript.
paper_table = friedman_results[
    ["Metric", "Friedman chi-square", "df", "p-value", "Kendall W"]
].copy()

paper_table["Metric"] = paper_table["Metric"].replace({
    "Precision": "Precision (Malignant)",
    "Recall": "Recall (Malignant)",
    "F1-score": "F1-score (Malignant)",
})

paper_table["Friedman chi-square"] = paper_table["Friedman chi-square"].map(lambda x: f"{x:.3f}")
paper_table["p-value"] = paper_table["p-value"].map(lambda x: f"{x:.3f}")
paper_table["Kendall W"] = paper_table["Kendall W"].map(lambda x: f"{x:.3f}")

paper_table

,Metric,Friedman chi-square,df,p-value,Kendall W
0,Accuracy,4.784,4,0.310,0.239
1,Precision (Malignant),3.680,4,0.451,0.184
2,Recall (Malignant),3.673,4,0.452,0.184
3,F1-score (Malignant),4.960,4,0.291,0.248
4,ROC-AUC,2.222,4,0.695,0.111


In [7]:
# Export the statistical table as CSV.
paper_table.to_csv("friedman_results.csv", index=False)
print("Saved: friedman_results.csv")

Saved: friedman_results.csv


In [8]:
# LaTeX rows for direct comparison with the manuscript table.
print(r"\begin{tabular}{lcccc}")
print(r"\hline")
print(r"\textbf{Metric} & $\boldsymbol{\chi^2}$ & \textbf{df} & $\boldsymbol{p}$\textbf{-value} & \textbf{Kendall's $W$} \\")
print(r"\hline")
for _, row in paper_table.iterrows():
    print(
        f"{row['Metric']} & {row['Friedman chi-square']} & {row['df']} & "
        f"{row['p-value']} & {row['Kendall W']} " + r"\\"
    )
print(r"\hline")
print(r"\end{tabular}")

\begin{tabular}{lcccc}
\hline
\textbf{Metric} & $\boldsymbol{\chi^2}$ & \textbf{df} & $\boldsymbol{p}$\textbf{-value} & \textbf{Kendall's $W$} \\
\hline
Accuracy & 4.784 & 4 & 0.310 & 0.239 \\
Precision (Malignant) & 3.680 & 4 & 0.451 & 0.184 \\
Recall (Malignant) & 3.673 & 4 & 0.452 & 0.184 \\
F1-score (Malignant) & 4.960 & 4 & 0.291 & 0.248 \\
ROC-AUC & 2.222 & 4 & 0.695 & 0.111 \\
\hline
\end{tabular}


In [9]:
# Post-hoc analysis is only performed when the omnibus Friedman test is significant.
# Holm correction is implemented directly to avoid an additional dependency.

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted_sorted = np.empty(m, dtype=float)

    running_max = 0.0
    for rank, idx in enumerate(order):
        adjusted = (m - rank) * p_values[idx]
        running_max = max(running_max, adjusted)
        adjusted_sorted[rank] = min(running_max, 1.0)

    adjusted = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        adjusted[idx] = adjusted_sorted[rank]
    return adjusted

pairwise_results = {}

for metric in metrics:
    omnibus_p = friedman_results.loc[
        friedman_results["Metric"] == metric, "p-value"
    ].iloc[0]

    if omnibus_p < 0.05:
        rows_pw = []
        for model_a, model_b in combinations(models, 2):
            a = df[df["Model"] == model_a].sort_values("Run")[metric].to_numpy()
            b = df[df["Model"] == model_b].sort_values("Run")[metric].to_numpy()

            stat, p = wilcoxon(a, b, alternative="two-sided", method="auto")
            rows_pw.append({
                "Model A": model_a,
                "Model B": model_b,
                "W": stat,
                "p-value": p,
            })

        pw = pd.DataFrame(rows_pw)
        pw["Holm-adjusted p"] = holm_adjust(pw["p-value"].to_numpy())
        pairwise_results[metric] = pw
        print(f"\nPost-hoc comparisons for {metric}:")
        display(pw)
    else:
        print(f"{metric}: Friedman p = {omnibus_p:.3f}; no post-hoc test required.")

Accuracy: Friedman p = 0.310; no post-hoc test required.
Precision: Friedman p = 0.451; no post-hoc test required.
Recall: Friedman p = 0.452; no post-hoc test required.
F1-score: Friedman p = 0.291; no post-hoc test required.
ROC-AUC: Friedman p = 0.695; no post-hoc test required.


## Interpretation

At $\alpha=0.05$, none of the five omnibus Friedman tests is statistically significant.

Therefore, the observed differences in mean performance are **descriptive differences** and should not be interpreted as evidence of statistical superiority.

This result does **not** demonstrate equivalence among the models. With only five matched runs, statistical power is limited; the current experiment simply does not provide sufficient evidence to reject the null hypothesis of equal performance ranks among the evaluated configurations.